In [3]:
from typing import Dict, List, Optional, Tuple
import json
import torch

class ProteinTokenizer:
    def __init__(
        self,
        vocab: Optional[Dict[str, int]] = None,
        special_tokens: Optional[List[str]] = None,
        pad_token: str = "[PAD]",
        unk_token: str = "[UNK]",
        bos_token: str = "[BOS]",
        eos_token: str = "[EOS]",
    ):
        if vocab is None:
            vocab = self._build_default_vocab(pad_token, unk_token, bos_token, eos_token, special_tokens or [])

        self.vocab = vocab
        self.inverse_vocab = {v: k for k, v in vocab.items()}

        self.pad_token = pad_token
        self.unk_token = unk_token
        self.bos_token = bos_token
        self.eos_token = eos_token

        self.pad_token_id = vocab[pad_token]
        self.unk_token_id = vocab[unk_token]
        self.bos_token_id = vocab[bos_token]
        self.eos_token_id = vocab[eos_token]

    def _build_default_vocab(self, pad_token: str, unk_token: str, bos_token: str, eos_token: str, extra_specials: List[str]) -> Dict[str, int]:
        # Standard amino acids + stop
        standard_aas = list("ACDEFGHIKLMNPQRSTVWY*")
        # Common ambiguous / special
        ambiguous = list("XBJZOU")

        base = standard_aas + ambiguous

        # Protein boundary markers
        boundaries = ["<PROT>", "</PROT>"]

        # Core special tokens
        specials = [
            bos_token, eos_token, pad_token, unk_token,
            "[MUT]", "[RESISTANT]", "[SUSCEPTIBLE]",
            "[CIPRO]", "[FLUOROQUINOLONE]", "[AMPC]", "[MEROPENEM]",  # antibiotics
            "[SPECIES_ECOLI]", "[PHYLO_REF]",
        ]

        # Add user-requested extras
        specials += extra_specials

        all_tokens = base + boundaries + specials

        vocab = {token: i for i, token in enumerate(all_tokens)}
        return vocab

    @property
    def vocab_size(self) -> int:
        return len(self.vocab)

    def encode(
        self,
        sequence: str,
        add_special_tokens: bool = True,
        conditioning: Optional[List[str]] = None,
    ) -> torch.LongTensor:
        """
        Encode a proteome string (with <PROT>...</PROT> markers)

        Args:
            sequence: concatenated proteome string
            add_special_tokens: whether to add [BOS]/[EOS]
            conditioning: list of conditioning tokens e.g. ["[SPECIES_ECOLI]", "[CIPRO]", "[RESISTANT]"]

        Returns:
            torch.LongTensor of token ids
        """
        tokens = []

        if add_special_tokens:
            tokens.append(self.bos_token_id)

        if conditioning:
            for cond in conditioning:
                if cond in self.vocab:
                    tokens.append(self.vocab[cond])
                else:
                    tokens.append(self.unk_token_id)

        # Character-level tokenization, but recognize full special tokens like <PROT>
        i = 0
        while i < len(sequence):
            # Check for known multi-char tokens first (greedy longest match)
            matched = False
            for special in ["<PROT>", "</PROT>"] + [t for t in self.vocab if t.startswith("[")]:
                if sequence[i:].startswith(special):
                    tokens.append(self.vocab[special])
                    i += len(special)
                    matched = True
                    break

            if not matched:
                aa = sequence[i].upper()
                tokens.append(self.vocab.get(aa, self.unk_token_id))
                i += 1

        if add_special_tokens:
            tokens.append(self.eos_token_id)

        return torch.tensor(tokens, dtype=torch.long)

    def encode_fast(
        self,
        sequence: str,
        add_special_tokens: bool = True,
        conditioning: Optional[List[str]] = None,
    ) -> torch.LongTensor:
        tokens = []

        if add_special_tokens:
            tokens.append(self.bos_token_id)

        if conditioning:
            for cond in conditioning:
                tokens.append(self.vocab.get(cond, self.unk_token_id))

        # Split on protein boundaries to avoid slow per-char checks
        parts = sequence.split("<PROT>")
        for part in parts:
            if not part.strip():
                continue
            subparts = part.split("</PROT>")
            for i, sub in enumerate(subparts):
                if i < len(subparts) - 1:  # before each </PROT>
                    # Add <PROT> only at start of real protein
                    if sub.strip():
                        tokens.append(self.vocab["<PROT>"])
                    # Tokenize AAs fast
                    for aa in sub.upper():
                        if aa in self.vocab:
                            tokens.append(self.vocab[aa])
                        else:
                            tokens.append(self.unk_token_id)
                    tokens.append(self.vocab["</PROT>"])
                else:
                    # Leftover after last </PROT> — rare, but handle
                    for aa in sub.upper():
                        tokens.append(self.vocab.get(aa, self.unk_token_id))

        if add_special_tokens:
            tokens.append(self.eos_token_id)

        return torch.tensor(tokens, dtype=torch.long)

    def decode(
        self,
        token_ids: torch.Tensor | List[int],
        skip_special: bool = False,
        remove_boundaries: bool = False,
    ) -> str:
        if isinstance(token_ids, torch.Tensor):
            token_ids = token_ids.tolist()

        tokens = []
        for tid in token_ids:
            token = self.inverse_vocab.get(tid, self.unk_token)
            if skip_special and token in {self.bos_token, self.eos_token, self.pad_token}:
                continue
            tokens.append(token)

        text = "".join(tokens)

        if remove_boundaries:
            text = text.replace("<PROT>", "").replace("</PROT>", "")

        return text

    def save(self, path: str):
        config = {
            "vocab": self.vocab,
            "pad_token": self.pad_token,
            "unk_token": self.unk_token,
            "bos_token": self.bos_token,
            "eos_token": self.eos_token,
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(config, f, indent=2)

    @classmethod
    def load(cls, path: str):
        with open(path, "r", encoding="utf-8") as f:
            config = json.load(f)
        return cls(
            vocab=config["vocab"],
            pad_token=config["pad_token"],
            unk_token=config["unk_token"],
            bos_token=config["bos_token"],
            eos_token=config["eos_token"],
        )


# ────────────────────────────────────────────────
# Quick test
# ────────────────────────────────────────────────

if __name__ == "__main__":
    tokenizer = ProteinTokenizer()

    print("Vocab size:", tokenizer.vocab_size)
    print("Some important ids:")
    print("  [BOS]:", tokenizer.bos_token_id)
    print("  [CIPRO]:", tokenizer.vocab.get("[CIPRO]"))
    print("  <PROT>:", tokenizer.vocab.get("<PROT>"))
    print("  A:", tokenizer.vocab.get("A"))

    # Example proteome snippet
    example = "<PROT>MATKTTTVNG</PROT><PROT>MFVKLLRSVA</PROT>"

    conditioning = ["[SPECIES_ECOLI]", "[CIPRO]", "[RESISTANT]"]

    encoded = tokenizer.encode(example, conditioning=conditioning)
    print("\nEncoded:", encoded.tolist())

    decoded = tokenizer.decode(encoded, skip_special=True)
    print("Decoded:", decoded)

    tokenizer.save("tokenizer.json")

Vocab size: 42
Some important ids:
  [BOS]: 29
  [CIPRO]: 36
  <PROT>: 27
  A: 0

Encoded: [29, 40, 36, 34, 27, 10, 0, 16, 8, 16, 16, 16, 17, 11, 5, 28, 27, 10, 4, 17, 8, 9, 9, 14, 15, 17, 0, 28, 30]
Decoded: [SPECIES_ECOLI][CIPRO][RESISTANT]<PROT>MATKTTTVNG</PROT><PROT>MFVKLLRSVA</PROT>


In [ ]:
import os
import sys
import pickle
import random
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

class ProteomeDataset(Dataset):
    def __init__(
        self,
        csv_path: str,
        tokenizer: ProteinTokenizer,
        chunk_size: int = 1024,
        overlap: int = 256,
        phylo_pkl: str = None,
        mode: str = "finetune",
        max_samples: int = None,
        start_idx: int = 0,
        use_mutated_only: bool = False,
    ):
        if phylo_pkl is None:
            phylo_pkl = "ecoli_phylo_distances.pkl"

        df = pd.read_csv(csv_path, dtype={'genome_id': str})

        # Optional: filter to only genomes with valid reversions (for finetuning)
        if use_mutated_only:
            if 'reversions_applied' in df.columns:
                df = df[df['reversions_applied'] == True].reset_index(drop=True)
                print(f"Filtered to {len(df)} genomes with reversions applied (mutated pairs)")
            else:
                print("Warning: 'reversions_applied' column not found — using all rows")

        # Optional subsample with start_idx for chunked loading
        if max_samples is not None:
            df = df.iloc[start_idx:start_idx + max_samples].reset_index(drop=True)
            print(f"Loading chunk: samples {start_idx} to {start_idx + len(df)}")
        else:
            df = df.iloc[start_idx:].reset_index(drop=True)

        self.df = df
        self.tokenizer = tokenizer
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.mode = mode
        self.stride = chunk_size - overlap

        # Load phylo matrix
        self.phylo_matrix = pickle.load(open(phylo_pkl, "rb"))
        self.num_reps = self.phylo_matrix.shape[0]

        self.genome_to_phylo_idx = {}
        for idx in range(len(self.df)):
            genome_id = self.df.iloc[idx]['genome_id']
            # Deterministic hash → fixed index
            h = int(hashlib.sha256(genome_id.encode()).hexdigest(), 16)
            phylo_idx = h % self.num_reps
            self.genome_to_phylo_idx[idx] = phylo_idx

        # Precompute chunk indices
        self.chunk_indices = []  # (genome_idx, start_pos)
        for genome_idx in range(len(self.df)):
            proteome_str = self.df.iloc[genome_idx]['unmutated_proteome']
            token_len = len(self.tokenizer.encode_fast(
                proteome_str, add_special_tokens=False, conditioning=None
            ))
            num_chunks = max(1, (token_len - chunk_size) // self.stride + 1)
            for chunk_num in range(num_chunks):
                start = chunk_num * self.stride
                self.chunk_indices.append((genome_idx, start))

        print(f"Dataset initialized: {len(self.df)} genomes → {len(self.chunk_indices)} chunks")

    def __len__(self):
        return len(self.chunk_indices)

    def __getitem__(self, idx):
        genome_idx, start = self.chunk_indices[idx]
        row = self.df.iloc[genome_idx]

        # Conditioning
        if self.mode == "finetune":
            cond = ["[SPECIES_ECOLI]", "[CIPRO]", "[RESISTANT]"]
            unmut_str = row['unmutated_proteome']
            mut_str   = row['mutated_proteome']

            # Concatenate: unmutated [SEP] mutated
            full_str = unmut_str + "[SEP]" + mut_str

            # Encode once
            full_ids = self.tokenizer.encode_fast(
                full_str,
                add_special_tokens=True,
                conditioning=cond
            )

            # Find separator position (for loss masking in model)
            sep_token_id = self.tokenizer.vocab.get("[SEP]", -1)
            sep_positions = (full_ids == sep_token_id).nonzero(as_tuple=True)[0]
            sep_pos = sep_positions[0].item() if len(sep_positions) > 0 else -1

            input_ids = full_ids
            labels    = full_ids.clone()

        else:  # pretrain
            cond = None
            input_str = row['unmutated_proteome']
            input_ids = self.tokenizer.encode_fast(
                input_str, add_special_tokens=True, conditioning=cond
            )
            labels = input_ids.clone()
            sep_pos = -1  # no masking

        # Chunk
        end = start + self.chunk_size
        input_chunk = input_ids[start:end]
        target_chunk = labels[start:end]

        # Pad if needed
        pad_len = self.chunk_size - len(input_chunk)
        if pad_len > 0:
            pad_tensor = torch.full((pad_len,), self.tokenizer.pad_token_id, dtype=torch.long)
            input_chunk = torch.cat([input_chunk, pad_tensor])
            target_chunk = torch.cat([target_chunk, pad_tensor])

        # Fixed phylo
        phylo_idx = self.genome_to_phylo_idx[genome_idx]
        phylo_dist = torch.tensor(self.phylo_matrix[phylo_idx], dtype=torch.float)

        return {
            'input_ids': input_chunk.long(),
            'labels': target_chunk.long(),
            'phylo_dist': phylo_dist,
            'genome_id': row['genome_id'],
            'sep_pos': sep_pos,  # used in model for loss masking
        }


def collate_fn(batch):
    return {
        'input_ids': torch.stack([b['input_ids'] for b in batch]),
        'labels': torch.stack([b['labels'] for b in batch]),
        'phylo_dist': torch.stack([b['phylo_dist'] for b in batch]),
        'genome_id': [b['genome_id'] for b in batch],
        'sep_pos': torch.tensor([b['sep_pos'] for b in batch]),  # (B,)
    }

In [ ]:
import math
from typing import Optional

import torch
import torch.nn as nn

class PhyloAttention(nn.Module):
    """
    Multi-head attention with:
    - ALiBi linear bias (no learned positional embeddings)
    - Phylogenetic distance bias (learned scalar weight)
    - Causal masking
    """
    def __init__(self, embed_dim: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        assert self.head_dim * num_heads == embed_dim, "embed_dim must be divisible by num_heads"

        self.scale = self.head_dim ** -0.5

        self.qkv_proj = nn.Linear(embed_dim, embed_dim * 3)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

        self.phylo_alpha = nn.Parameter(torch.tensor(0.5))

    def forward(
        self,
        x: torch.Tensor,                    # (B, L, D)
        phylo_dists: torch.Tensor | None,   # (B, num_reps) or (B, L, L), or None
        alibi_bias: torch.Tensor,           # (num_heads, L, L)
        attn_mask: Optional[torch.Tensor] = None,  # (1, 1, L, L) causal
    ):
        B, L, D = x.shape

        qkv = self.qkv_proj(x).reshape(B, L, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)          # (3, B, heads, L, head_dim)
        q, k, v = qkv.unbind(0)

        # Scale attention temperature by phylogenetic distance; clamp keeps temp positive.
        if phylo_dists is not None:
            if phylo_dists.dim() == 2:             # (B, num_reps)
                phylo_scalar = phylo_dists.mean(dim=-1).view(B, 1, 1, 1)
            else:                                  # (B, L, L)
                phylo_scalar = phylo_dists.mean(dim=(-1, -2)).view(B, 1, 1, 1)
            temp = (1.0 + self.phylo_alpha * phylo_scalar).clamp(min=1e-6)
        else:
            temp = 1.0

        scores = (q @ k.transpose(-2, -1)) * (self.scale * temp)  # (B, heads, L, L)
        scores = scores + alibi_bias.unsqueeze(0)                  # ALiBi positional bias

        if attn_mask is not None:
            scores = scores.masked_fill(attn_mask == 0, float('-inf'))

        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        out = attn @ v                             # (B, heads, L, head_dim)
        out = out.transpose(1, 2).reshape(B, L, D)
        return self.out_proj(out)


class PhyloGenBlock(nn.Module):
    def __init__(
        self,
        embed_dim: int,
        num_heads: int,
        ff_dim: int = None,
        dropout: float = 0.1,
    ):
        super().__init__()
        ff_dim = ff_dim or embed_dim * 4

        self.attn = PhyloAttention(embed_dim, num_heads, dropout=dropout)

        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Linear(ff_dim, embed_dim),
            nn.Dropout(dropout),
        )

        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x, phylo_dists, alibi_bias, attn_mask):
        attn_out = self.attn(self.norm1(x), phylo_dists, alibi_bias, attn_mask)
        x = x + attn_out
        ffn_out = self.ffn(self.norm2(x))
        x = x + ffn_out
        return x


class ALiBiEmbedder(nn.Module):
    def __init__(
        self,
        vocab_size: int = 9,
        embed_dim: int = 256,
        max_len: int = 1024,
        dropout: float = 0.1,
        num_heads: int = 8,
    ):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.max_len = max_len

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.dropout = nn.Dropout(dropout)

        slopes = self._get_alibi_slopes(num_heads)
        self.register_buffer("slopes", slopes)

        bias_matrix = self._build_alibi_bias_matrix(max_len, slopes)
        self.register_buffer("bias_matrix", bias_matrix)

    def _get_alibi_slopes(self, num_heads: int) -> torch.Tensor:
        def get_slopes_power_of_2(n):
            start = 2 ** (-(2 ** -(math.log2(n) - 3)))
            ratio = start
            return [start * (ratio**i) for i in range(n)]

        if math.log2(num_heads).is_integer():
            slopes = get_slopes_power_of_2(num_heads)
        else:
            closest_power_of_2 = 2 ** math.floor(math.log2(num_heads))
            slopes = get_slopes_power_of_2(closest_power_of_2)
            extra_slopes_count = num_heads - closest_power_of_2
            extra_base = 2 ** (-(2 ** -(math.log2(2 * closest_power_of_2) - 3)))
            extra_slopes = [extra_base * (extra_base**i) for i in range(extra_slopes_count)]
            slopes = slopes + extra_slopes

        return torch.tensor(slopes, dtype=torch.float32)

    def _build_alibi_bias_matrix(self, max_len: int, slopes: torch.Tensor) -> torch.Tensor:
        positions = torch.arange(max_len).unsqueeze(1)
        distances = torch.abs(positions - positions.T)
        biases = -slopes.unsqueeze(1).unsqueeze(2) * distances.unsqueeze(0)
        return biases

    def get_alibi_bias(self, seq_len: int) -> torch.Tensor:
        if seq_len <= self.max_len:
            return self.bias_matrix[:, :seq_len, :seq_len]
        else:
            return self._build_alibi_bias_matrix(seq_len, self.slopes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.embedding(x) * math.sqrt(self.embed_dim))


class PhyloGen(nn.Module):
    """
    Decoder-only transformer with:
    - Token embedding
    - ALiBi positional bias
    - Phylogenetic attention bias
    - Standard transformer blocks
    """
    def __init__(
        self,
        vocab_size: int,
        tokenizer=None,
        embed_dim: int = 256,
        num_heads: int = 8,
        num_layers: int = 6,
        max_seq_len: int = 2048,
        dropout: float = 0.1,
    ):
        super().__init__()

        self.embed_dim = embed_dim
        self.num_heads = num_heads

        if tokenizer is not None:
            self.pad_token_id = tokenizer.pad_token_id
        else:
            self.pad_token_id = 31

        self.token_embed = nn.Embedding(vocab_size, embed_dim, padding_idx=self.pad_token_id)
        self.embed_dropout = nn.Dropout(dropout)

        self.alibi = ALiBiEmbedder(
            vocab_size=vocab_size,
            embed_dim=embed_dim,
            max_len=max_seq_len,
            num_heads=num_heads,
            dropout=dropout,
        )

        self.blocks = nn.ModuleList([
            PhyloGenBlock(embed_dim=embed_dim, num_heads=num_heads, dropout=dropout)
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size, bias=False)

        self._init_weights()

        # Weight tying
        self.lm_head.weight = self.token_embed.weight

    def _init_weights(self):
        nn.init.normal_(self.token_embed.weight, mean=0.0, std=0.02)
        nn.init.normal_(self.lm_head.weight, mean=0.0, std=0.02 / math.sqrt(self.embed_dim))

    def forward(
        self,
        input_ids: torch.Tensor,
        phylo_dists: torch.Tensor,
        labels: Optional[torch.Tensor] = None,
        sep_pos: Optional[torch.Tensor] = None,
        return_dict: bool = True,
    ):
        B, L = input_ids.shape

        x = self.token_embed(input_ids) * math.sqrt(self.embed_dim)
        x = self.embed_dropout(x)

        alibi_bias = self.alibi.get_alibi_bias(L)
        attn_mask = torch.tril(torch.ones(L, L, device=x.device, dtype=torch.bool)).view(1, 1, L, L)

        for block in self.blocks:
            x = block(x, phylo_dists, alibi_bias, attn_mask)

        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()

            if sep_pos is not None:
                ignore_mask = torch.zeros_like(shift_labels, dtype=torch.bool)
                for b in range(B):
                    pos = sep_pos[b].item()
                    if pos >= 0:
                        ignore_mask[b, :pos] = True
                shift_labels = shift_labels.masked_fill(ignore_mask, -100)

            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fct(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1)
            )

        if return_dict:
            return {"logits": logits, "loss": loss}
        return logits

    def to(self, *args, **kwargs):
        super().to(*args, **kwargs)
        self.alibi.to(*args, **kwargs)
        return self

In [6]:
import sys
import torch
import json

from torch.utils.data import DataLoader
import torch.multiprocessing as mp
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")   
print(f"Using device: {device}")

tokenizer_path = "tokenizer.json"
tokenizer = ProteinTokenizer.load(str(tokenizer_path))

csv_path = "ecoli_pairs_concat.csv"
phylo_pkl_path = "ecoli_phylo_distances.pkl"

epochs = 3
batch_size = 32
chunk_size = 1024
overlap = 256
lr = 3e-4
max_samples = None
use_mutated_only = False
checkpoint_dir = Path("checkpoints_pretrain")
checkpoint_dir.mkdir(exist_ok=True, parents=True)

loss_log_file = Path("pretrain_loss_log.json")

save_every_steps = 5000

resume_from = None

Using device: cuda


In [7]:
model = PhyloGen(
    vocab_size=tokenizer.vocab_size,
    tokenizer=tokenizer,
    embed_dim=256,
    num_heads=8,
    num_layers=6,
    max_seq_len=2048,
).to(device)

model = torch.compile(model)

In [8]:
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)

start_epoch = 0
global_step = 0
current_epoch_steps = 0
loss_log = []

if loss_log_file.exists():
    with open(loss_log_file, "r") as f:
        loss_log = json.load(f)
else:
    loss_log = [{}]

print(loss_log)

if resume_from:
    print(f"Resuming from checkpoint: {resume_from}")
    ckpt = torch.load(resume_from, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    
    start_epoch = ckpt.get('epoch', 1) - 1
    global_step = ckpt.get('step', 0)
    
    print(f"Resumed at epoch {start_epoch + 1} (1-based), global step {global_step}")
else:
    print("Starting fresh pretraining")


[{}]
Starting fresh pretraining


In [10]:
dataset = ProteomeDataset(
    csv_path=str(csv_path),
    tokenizer=tokenizer,
    chunk_size=chunk_size,
    overlap=overlap,
    phylo_pkl=str(phylo_pkl_path),
    mode="pretrain",
    max_samples=max_samples,
    use_mutated_only=use_mutated_only,
)

Dataset initialized: 469 genomes → 778448 chunks


In [11]:
loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=6,
    prefetch_factor=2,
    persistent_workers=True,
    pin_memory=True,
)

In [ ]:
model.train()

for epoch in range(start_epoch, epochs):
    epoch_num = epoch + 1

    if resume_from and epoch == start_epoch:
        print(f"\nContinuing pretrain Epoch {epoch_num}/{epochs} from step {global_step}")
    else:
        print(f"\nPretrain Epoch {epoch_num}/{epochs}")

    total_loss = 0
    steps_this_epoch = 0

    pbar = tqdm(loader, desc=f"Epoch {epoch_num}/{epochs}")
    for batch in pbar:
        global_step += 1
        steps_this_epoch += 1

        input_ids = batch['input_ids'].to(device)
        phylo = batch['phylo_dist'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        with torch.amp.autocast(device_type=device.type):
            out = model(input_ids, phylo, labels=labels)
            loss = out["loss"]

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

        loss_log.append({
            "epoch": epoch_num,
            "step": global_step,
            "loss": loss.item(),
            "avg_loss_this_epoch": total_loss / steps_this_epoch if steps_this_epoch > 0 else 0,
        })

        if global_step % save_every_steps == 0:
            ckpt_path = checkpoint_dir / f"pretrain_step_{global_step}.pt"
            torch.save({
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'step': global_step,
                'epoch': epoch_num,
            }, ckpt_path)
            print(f"\nCheckpoint saved: {ckpt_path}")

            with open(loss_log_file, "w") as f:
                json.dump(loss_log, f, indent=4)

    avg_loss = total_loss / steps_this_epoch if steps_this_epoch > 0 else 0
    print(f"Epoch {epoch_num} finished | Avg loss: {avg_loss:.4f} | Steps this epoch: {steps_this_epoch}")

    ckpt_path = checkpoint_dir / f"pretrain_epoch_{epoch_num}.pt"
    torch.save({
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'step': global_step,
        'epoch': epoch_num,
    }, ckpt_path)
    print(f"Epoch checkpoint saved: {ckpt_path}")

    with open(loss_log_file, "w") as f:
        json.dump(loss_log, f, indent=4)

print("Pretraining complete.")


Pretrain Epoch 1/3


Epoch 1/3:   0%|          | 0/24327 [00:00<?, ?it/s]